## 6.2 PINN：正问题与反问题

### 方法与证据

#### 1. Poisson解析解、端点不可辨识与数值残差

A2模型为

$$
-ku''(x)=\pi^2\sin(\pi x),\qquad
u(0)=u(1)=0,\qquad k>0.
$$

两次积分得到

$$
u(x;k)=\frac{\sin(\pi x)}{k}.
$$

验证条件：取$k=0.75,1.5,3$，在1001点上评价残差；最大绝对残差小于$10^{-12}$，端点位移为零。不同$k$对应相同端点观测，构成不可辨识反例。

In [1]:
import math
import numpy as np

K_TRUE = 1.5
SEEDS = (2026, 2027, 2028)
OBS_X = np.arange(1, 13, dtype=float) / 13.0
NOISE_SEED = 6102
RTOL_ANALYTIC = 1e-12

def exact_u(x, k=K_TRUE):
    return np.sin(np.pi * np.asarray(x, dtype=float)) / k

def exact_uxx(x, k=K_TRUE):
    return -(np.pi**2)*np.sin(np.pi*np.asarray(x, dtype=float))/k

def exact_sensitivity_k(x, k=K_TRUE):
    return -np.sin(np.pi*np.asarray(x, dtype=float))/k**2

def observations(sigma):
    # 与A2保存代码一致：每个噪声水平重新用同一固定种子生成标准正态向量。
    noise = np.random.default_rng(NOISE_SEED).standard_normal(len(OBS_X))
    return exact_u(OBS_X) + sigma * noise

def estimate_ls(x, y):
    s = np.sin(np.pi*np.asarray(x))
    bhat = np.dot(s, y)/np.dot(s, s)
    if bhat <= 0:
        raise ValueError("幅值估计非正，不能解释为有限正系数")
    return 1.0/bhat

grid=np.linspace(0.0,1.0,1001)
for k in (0.75,1.5,3.0):
    residual=-k*exact_uxx(grid,k)-np.pi**2*np.sin(np.pi*grid)
    assert np.max(np.abs(residual)) < RTOL_ANALYTIC
    assert np.allclose(exact_u([0.0,1.0],k),[0.0,0.0],atol=RTOL_ANALYTIC)
    print(f"k={k:g}; max(PDE residual)={np.max(abs(residual)):.3e}; u(0.5)={float(exact_u(0.5,k)):.9f}")
print("端点观测均为0：不同k不可由端点位移区分。")


k=0.75; max(PDE residual)=8.882e-16; u(0.5)=1.333333333
k=1.5; max(PDE residual)=8.882e-16; u(0.5)=0.666666667
k=3; max(PDE residual)=8.882e-16; u(0.5)=0.333333333
端点观测均为0：不同k不可由端点位移区分。


#### 2. 物理损失、Softplus链式梯度和最小二乘

令测试函数$u_{\mathrm{test}}(x)=0.6\sin(\pi x)$，取$k=1.3$。对残差均方损失

$$
L_f(k)=\frac1N\sum_i
\bigl(-ku_{\mathrm{test}}''(x_i)-\pi^2\sin(\pi x_i)\bigr)^2
$$

分别计算解析梯度及步长$10^{-5}$的中心差分梯度，要求归一化差小于$10^{-6}$。同时测试

$$
k=\log(1+e^\rho),\qquad
\frac{\partial L_f}{\partial\rho}
=\frac{\partial L_f}{\partial k}\frac1{1+e^{-\rho}}.
$$

解析最小二乘以12个观测点$x_j=j/13$、真实$k_*=1.5$、噪声种子6102和$\sigma=0,0.01$计算。其参照值分别是1.5和1.525781004180，容差$10^{-12}$。

In [2]:
c = 0.6
xx = np.linspace(0.02,0.98,64)
s = np.sin(np.pi*xx)
def residual_test(k):
    return (k*c-1.0)*np.pi**2*s
def lf_test(k):
    return float(np.mean(residual_test(k)**2))
def dlf_dk_analytic(k):
    uxx = -c*np.pi**2*s
    return float(-2*np.mean(residual_test(k)*uxx))

k0=1.3
h=1e-5
grad_fd=(lf_test(k0+h)-lf_test(k0-h))/(2*h)
grad_exact=dlf_dk_analytic(k0)
print(f"∂Lf/∂k: analytic={grad_exact:.12f}; finite_difference={grad_fd:.12f}")
assert abs(grad_exact-grad_fd)/max(1,abs(grad_exact)) < 1e-6

def softplus(rho):
    return np.logaddexp(0.0,rho)
def sigmoid(rho):
    return 1.0/(1.0+np.exp(-rho))
rho0=np.log(np.expm1(k0))
assert abs(softplus(rho0)-k0)<1e-12
grad_rho=dlf_dk_analytic(softplus(rho0))*sigmoid(rho0)
grad_rho_fd=(lf_test(softplus(rho0+h))-lf_test(softplus(rho0-h)))/(2*h)
print(f"∂Lf/∂rho: analytic={grad_rho:.12f}; finite_difference={grad_rho_fd:.12f}")
assert abs(grad_rho-grad_rho_fd)/max(1,abs(grad_rho)) < 1e-6

for sigma, expected in [(0.,1.5),(.01,1.5257810041800255)]:
    y=observations(sigma)
    estimate=estimate_ls(OBS_X,y)
    print(f"sigma={sigma:.2f}, analytic LS k={estimate:.12f}, A2 saved={expected:.12f}")
    assert abs(estimate-expected)<1e-12
print("独立计算通过：参数梯度、链式梯度、正性初值和固定噪声LS。")


∂Lf/∂k: analytic=-13.184268099027; finite_difference=-13.184268099176
∂Lf/∂rho: analytic=-9.591135874158; finite_difference=-9.591135873821
sigma=0.00, analytic LS k=1.500000000000, A2 saved=1.500000000000
sigma=0.01, analytic LS k=1.525781004180, A2 saved=1.525781004180
独立计算通过：参数梯度、链式梯度、正性初值和固定噪声LS。


#### 3. B2灵敏度扰动反例

B2控制方程为

$$
-u''(x)+a\,u(x)=f(x),\qquad a>0,
\qquad u(0)=u(1)=0.
$$

设$w=\partial u/\partial a$，则

$$
-w''+a_*w=-u^*,\qquad
\delta u\approx w\,\delta a.
$$

若$v=\delta u$，就不能再次写$\delta u=v\,\delta a$。令$a_*=1$、$u^*=\sin(\pi x)$、$\delta a=10^{-4}$，比较精确扰动与两种预测；要求正确的一阶预测相对偏差小于$10^{-4}$。

In [3]:
A_REACTION=1.0
def u_B(x,a):
    # 固定 f=(pi^2+1)sin(pi x)；改变的是反应系数a
    return (np.pi**2 + A_REACTION)/(np.pi**2+a)*np.sin(np.pi*np.asarray(x))
def w_B(x):
    return -np.sin(np.pi*np.asarray(x))/(np.pi**2+A_REACTION)

for x in (0.0,0.001,0.5,1.0):
    print(f"x={x:g}; ∂u/∂a={float(w_B(x)):.12g}")
assert abs(float(w_B(0.0))) < 1e-14
assert abs(float(w_B(1.0))) < 1e-14

delta_a=1e-4
actual=float(u_B(0.5,A_REACTION+delta_a)-u_B(0.5,A_REACTION))
correct=float(w_B(0.5))*delta_a
wrong=correct*delta_a  # B2文字如把v=delta u再次乘delta_a的后果
print(f"delta_a={delta_a:g}, actual delta_u={actual:.12e}")
print(f"linear delta_u=w*delta_a={correct:.12e}; duplicated delta_a -> {wrong:.12e}")
assert abs(actual-correct)/abs(actual) < 1e-4
assert abs(wrong) < abs(actual)/1000
print("B2扰动符号反例通过；未将其误称为A2方程的错误。")


x=0; ∂u/∂a=-0
x=0.001; ∂u/∂a=-0.000289025006795
x=0.5; ∂u/∂a=-0.0919996683504
x=1; ∂u/∂a=-1.12667099368e-17
delta_a=0.0001, actual delta_u=-9.199882196387e-06
linear delta_u=w*delta_a=-9.199966835038e-06; duplicated delta_a -> -9.199966835038e-10
B2扰动符号反例通过；未将其误称为A2方程的错误。


#### 4. B2源项振幅未知时的可辨识性

设$f(x)=c\sin(\pi x)$，其中$a,c$未知，则

$$
u(x)=\frac{c}{\pi^2+a}\sin(\pi x).
$$

取$(a_1,c_1)=(1,\pi^2+1)$、$(a_2,c_2)=(2,\pi^2+2)$，检验1001个等距点上的两组状态是否相同。最大绝对差判据为小于$10^{-12}$。

In [4]:
xx=np.linspace(0,1,1001)
def u_source_unknown(x,a,c):
    return c/(np.pi**2+a)*np.sin(np.pi*np.asarray(x))
u1=u_source_unknown(xx,1.0,np.pi**2+1)
u2=u_source_unknown(xx,2.0,np.pi**2+2)
max_diff=float(np.max(np.abs(u1-u2)))
print("两组不同(a,c)产生相同状态；max|u1-u2| =",f"{max_diff:.3e}")
assert max_diff<1e-12
print("可识别组合仅为 c/(pi^2+a)，除非另有独立信息。")


两组不同(a,c)产生相同状态；max|u1-u2| = 0.000e+00
可识别组合仅为 c/(pi^2+a)，除非另有独立信息。


#### 5. A2保存结果的统计核对

统计数据为A2第6.2节已保存的反演输出，**仅复算统计量，不代表重新训练PINN**。对每种噪声分别给出三种启动条件的系数相对误差最小值、中位数、最大值及比值；并核对软边界诊断的端点误差。

In [5]:
import numpy as np
saved_inverse = [(2026, 0.0, 1.5001078210724295, 7.188071495297284e-05),
 (2027, 0.0, 1.5003076563552327, 0.0002051042368217774),
 (2028, 0.0, 1.5000283914815804, 1.8927654386935444e-05),
 (2026, 0.01, 1.5264019948178071, 0.017601329878538092),
 (2027, 0.01, 1.5265155318807988, 0.01767702125386587),
 (2028, 0.01, 1.526164428438928, 0.017442952292618646)]
saved_weak = [(2026, 1.5429298140225778, 0.008424717554609412, 0.00972758727212793),
 (2027, 1.5429545785401158, 0.008474399109190841, 0.009726370384203657),
 (2028, 1.5430065600763374, 0.00847630170476919, 0.00973007371114663)]

for sigma in (0.0,0.01):
    errors=sorted(100*r[3] for r in saved_inverse if r[1]==sigma)
    assert len(errors)==3
    print(f'sigma={sigma:g}: min={errors[0]:.9f}%, median={errors[1]:.9f}%, max={errors[2]:.9f}%, ratio={errors[2]/errors[0]:.5f}')
for seed,k,boundary,rmse in saved_weak:
    print(f'weak boundary seed={seed}: k={k:.6f}, boundary={boundary:.6f}, obs_RMSE={rmse:.6f}')
    assert boundary>0.001


sigma=0: min=0.001892765%, median=0.007188071%, max=0.020510424%, ratio=10.83622
sigma=0.01: min=1.744295229%, median=1.760132988%, max=1.767702125%, ratio=1.01342
weak boundary seed=2026: k=1.542930, boundary=0.008425, obs_RMSE=0.009728
weak boundary seed=2027: k=1.542955, boundary=0.008474, obs_RMSE=0.009726
weak boundary seed=2028: k=1.543007, boundary=0.008476, obs_RMSE=0.009730


### AI线索、代码来源与人工判断

| 项目 | 来源 | 核验结果 | 判断 |
| --- | --- | --- | --- |
| A2解析解与系数梯度 | A2第6.2节正文 | 解析残差与中心差分梯度通过 | 已核对；PINN训练未重跑 |
| A2最小二乘 | A2解析基准 | 无噪声1.5；含噪1.525781004180 | 解析计算通过 |
| B2灵敏度变量 | B2推导4 | 解扰动重复乘入参数扰动；解析反例通过 | B2需更正变量定义 |
| B2未知源项振幅 | B2推导5 | 两组不同参数产生同一状态 | 可识别组合为$c/(\pi^2+a)$ |
| B2多种子预期 | B2推导6、A2保存数据 | 无噪声最大/最小约10.836，含噪约1.013 | “可相差”不是“必然相差” |

**AI线索：** B2在S3中转述了以灵敏度范数解释噪声放大的AI建议；原始提示词和逐字回复未提供，不能确认其完整适用条件。其他B2数学表述问题尚无原始AI来源证据。

**代码来源：** 上述解析验证代码由AI辅助生成，已执行并保留输出。人工修改与核验签名待补。

**结果范围：** 解析测试运行完成，A2保存输出已重算；A2的PINN训练、B2原验证Notebook均未重新运行。

## 6.4 AI关联：科学基础模型

### 方法与证据

#### 测试1：Poisson解析解、零边界和唯一解族

模型为

$$
-ku''(x)=\sum_{m=1}^3c_m\sin(m\pi x),
\quad u(0)=u(1)=0,\quad k>0.
$$

解析解：

$$
u(x)=\sum_{m=1}^3\frac{c_m}{k(m\pi)^2}\sin(m\pi x).
$$

输入$k=1.7$、$c=(0.8,0.3,-0.6)$，以4097个评价点验证解析方程残差、两端边界和结果唯一性所需的解族结构。预期误差小于$10^{-12}$。

In [6]:
import numpy as np
k=1.7
c=np.array([0.8,0.3,-0.6],dtype=float)
m=np.arange(1,4,dtype=float)
x=np.linspace(0.0,1.0,4097)
S=np.sin(np.pi*np.outer(x,m))
f=S@c
u_exact=S@(c/(k*(m*np.pi)**2))
u_exact_xx=-S@(c/k)
r_exact=-k*u_exact_xx-f
err_res=np.max(np.abs(r_exact))
err_bc=max(abs(u_exact[0]),abs(u_exact[-1]))
assert err_res<1e-12 and err_bc<1e-12
print(f'Poisson exact: residual max={err_res:.3e}, boundary max={err_bc:.3e}')
print(f'u_exact(0.5)={u_exact[len(x)//2]:.12f}')


Poisson exact: residual max=2.220e-16, boundary max=3.285e-18
u_exact(0.5)=0.051653936759


#### 测试2：固定正弦基模型（6.4-3）、物理残差（6.4-4）

$$
\widehat u_\theta(x)=\frac{1}{\pi^2}
\sum_{m=1}^3b_m\sin(m\pi x),
\quad
r_\theta(x)=\sum_{m=1}^3(km^2b_m-c_m)\sin(m\pi x).
$$

任选预测系数$b=(0.4,-0.1,0.08)$，用解析二阶导与中心差分（步长$h=10^{-4}$）比较残差。解析与差分最大误差阈值$10^{-5}$；验证两端硬边界误差小于$10^{-12}$。

In [7]:
b=np.array([0.4,-0.1,0.08],dtype=float)
def pred(points, coeff=b):
    pp=np.asarray(points)
    return np.sin(np.pi*np.outer(pp,m))@coeff/(np.pi**2)
u_pred=pred(x)
d=k*(m**2)*b-c
r_modal=S@d
u_pred_xx=-S@((m**2)*b)
r_diff=-k*u_pred_xx-f
h=1e-4
u_second_fd=(pred(x[1:-1]+h)-2*pred(x[1:-1])+pred(x[1:-1]-h))/h**2
r_fd=-k*u_second_fd-f[1:-1]
diff_analytic=np.max(np.abs(r_modal-r_diff))
diff_fd=np.max(np.abs(r_modal[1:-1]-r_fd))
boundary=max(abs(u_pred[0]),abs(u_pred[-1]))
assert diff_analytic<1e-12 and diff_fd<1e-5 and boundary<1e-12
print(f'fixed basis: analytic residual difference={diff_analytic:.3e}')
print(f'centered finite difference residual difference={diff_fd:.3e}, boundary max={boundary:.3e}')


fixed basis: analytic residual difference=8.882e-16
centered finite difference residual difference=1.194e-07, boundary max=1.042e-17


#### 测试3：命题6.4-1（6.4-5）的连续范数上界及等号条件

令$d_m=km^2b_m-c_m$，则

$$
\|e\|_{L^2}^2=\frac12\sum_{m=1}^3
\frac{d_m^2}{k^2(m\pi)^4},
\qquad
\|r\|_{L^2}^2=\frac12\sum_{m=1}^3d_m^2.
$$

因此$\|e\|_{L^2}\le\|r\|_{L^2}/(k\pi^2)$。在独立4097点网格上用梯形法与精确模态平方和互相核验，容差$10^{-10}$；并构造仅第一模态有误差的等号情形。

In [8]:
def norm2(values):
    return float(np.sqrt(np.trapezoid(np.asarray(values)**2,x)))
e_pred=u_pred-u_exact
r_pred=r_modal
e_grid=norm2(e_pred)
r_grid=norm2(r_pred)
e_closed=np.sqrt(.5*np.sum((d/(k*(m*np.pi)**2))**2))
r_closed=np.sqrt(.5*np.sum(d**2))
upper=r_closed/(k*np.pi**2)
assert abs(e_grid-e_closed)<1e-10
assert abs(r_grid-r_closed)<1e-10
assert e_grid<=upper+1e-12
print(f'bound: error={e_grid:.12f}, residual={r_grid:.12f}, upper={upper:.12f}')

b_equal=c/(k*m**2)
b_equal[0]+=0.2/k
d_equal=k*m**2*b_equal-c
e_equal=S@(d_equal/(k*(m*np.pi)**2))
r_equal=S@d_equal
assert abs(norm2(e_equal)-norm2(r_equal)/(k*np.pi**2))<1e-12
print(f'equality first mode: error={norm2(e_equal):.12f}, upper={norm2(r_equal)/(k*np.pi**2):.12f}')


bound: error=0.014322704428, residual=1.466590604088, upper=0.087409821132
equality first mode: error=0.008428811298, upper=0.008428811298


#### 测试4：遗漏第四模态的下界（6.4-6）

在相同条件下把源项扩展为四个正弦模态，令$c_4=0.75$，预测模型仍只有前三个模态。若前三模态拟合精确，则剩余误差恰好为第四模态：

$$
\|e\|_{L^2(0,1)}
=\frac{|c_4|}{16\sqrt2\,k\pi^2}.
$$

以数值积分验证下界等号，容差$10^{-12}$。再证明残差RMS排序不能保证解误差排序：$e_A=\sin(\pi x)$、$e_B=\tfrac12\sin(3\pi x)$。

In [9]:
c4=0.75
u4=c4/(16*k*np.pi**2)*np.sin(4*np.pi*x)
u_four=u_exact+u4
e_missing=u_exact-u_four
lower=abs(c4)/(16*np.sqrt(2)*k*np.pi**2)
assert abs(norm2(e_missing)-lower)<1e-12
assert lower>0
print(f'missing 4th mode: error={norm2(e_missing):.12f}, lower={lower:.12f}')

err_A=np.sin(np.pi*x)
err_B=0.5*np.sin(3*np.pi*x)
res_A=np.pi**2*np.sin(np.pi*x)
res_B=0.5*(3*np.pi)**2*np.sin(3*np.pi*x)
assert norm2(err_A)>norm2(err_B) and norm2(res_A)<norm2(res_B)
print(f'ordering: error A={norm2(err_A):.6f} > B={norm2(err_B):.6f}')
print(f'ordering: residual A={norm2(res_A):.6f} < B={norm2(res_B):.6f}')


missing 4th mode: error=0.001975502648, lower=0.001975502648
ordering: error A=0.707107 > B=0.353553
ordering: residual A=6.978864 < B=31.404889


#### 测试5：A2迁移实验保存指标的静态算术检查

A2表6.4-4报告目标范围与外推范围上A/B/C/D四方案的平均相对$L^2$误差和平均残差RMS。下面**仅根据教材所列数值**计算差异，不运行源任务预训练、微调或Poseidon模型。

In [10]:
saved={
  'target': {
      'A':(54.41,0.5662,0), 'B':(6.77,0.1019,84),
      'C':(5.83,0.0855,100), 'D':(14.27,0.3130,27)},
  'extrapolation': {
      'A':(210.87,1.6278,0), 'B':(57.34,0.5268,0),
      'C':(55.12,0.4944,0), 'D':(40.88,0.7807,0)}
}
assert saved['target']['C'][0]<saved['target']['D'][0]
assert saved['extrapolation']['C'][0]>saved['extrapolation']['D'][0]
assert saved['extrapolation']['C'][1]<saved['extrapolation']['D'][1]
target_advantage=saved['target']['D'][0]-saved['target']['C'][0]
far_disadvantage=saved['extrapolation']['C'][0]-saved['extrapolation']['D'][0]
print(f'target: D-C error={target_advantage:.2f} percentage points (C lower)')
print(f'far extrapolation: C-D error={far_disadvantage:.2f} percentage points (C higher)')
print(f'far residual: C={saved["extrapolation"]["C"][1]:.4f} < D={saved["extrapolation"]["D"][1]:.4f}')
print(f'target C: 100/192 passed, {192-saved["target"]["C"][2]}/192 not jointly passing')
print('These are A2 stored table values, not retrained results.')


target: D-C error=8.44 percentage points (C lower)
far extrapolation: C-D error=14.24 percentage points (C higher)
far residual: C=0.4944 < D=0.7807
target C: 100/192 passed, 92/192 not jointly passing
These are A2 stored table values, not retrained results.


### AI线索、代码来源与人工判断

- **AI线索：** ChatGPT（GPT-6）依据A2 6.4原文辅助重新推导模型与命题。用户实际请求包括“将6.4节证明，公式，模型等重新推导”。B2的AI辅助对比S1—S5均在6.1—6.3范围，**未提供B2针对6.4的AI回复原话**；V6/B003是B2报告的文件存在性问题而非AI幻觉。
- **代码来源：** 本Notebook的解析校验代码由ChatGPT辅助编写，采用NumPy实现，五个代码单元已实际执行并保存标准输出；没有训练神经网络，也没有模拟训练日志。
- **人工修改/签核：** 未提供C组成员修改代码、独立手算和签名记录；待填写。
- **判断依据：** 式（6.4-3）、（6.4-4）、（6.4-5）、（6.4-6）的解析恒等式、正交性及数值积分互相一致。式（6.4-1）和（6.4-2）是经验优化目标而非全局收敛证明。
- **候选状态：** “有限点残差等同全域误差界”“小模型复现基础模型”“源范围以外的目标适配等同未见范围外推”均为教学警示，A2原文本身已经作出适用范围限定，不能直接登记成已确认的A2 AI错误。
- **B2交叉结论：** B2独立推导10仅限定其文献/来源检查范围；其V6报告A2引用五个数据文件缺失（B003），尚未完成C组仓库级复测；B2在6.4章稿选用上倾向A1、待A2补齐数据后再比较，**不代表C组已经独立核验A1**。
- **未验证事项：** B2原验证代码与B003五项具体路径、A2 6.4训练的独立复跑、完整训练数据划分和权重、Poseidon论文原文逐页查验、学生手算签核、最终整合稿。